# Project Milestone 1

<b> Project Overview</b><br>
Business problem draft<br>
Intended user or decision context<br>
Dataset description and source link<br>
List of raw files or tables<br>
Explanation of what one row means in each important table<br>
Initial row and column counts<br>
Data types<br>
Missing values<br>
Duplicate records or duplicate keys<br>
Suspicious, impossible, or confusing values<br>
Initial cleaning plan<br>
Potential target or outcome variable<br>
Early ethics, privacy, governance, or responsible-use concerns<br>


In [1]:
#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

/usr/bin/python3
3.13.5 (main, Aug 10 2026, 12:06:59) [GCC 14.2.0]


---
# Loading The Data <br>

The initial step in this is project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [2]:
# This code loads libraries and sets directors for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

#Set repository root to the current working directory. One step above readme location.
display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

#set max column display to 100 rows for pandas dataframes.
pd.set_option("display.max_columns", 100)

PosixPath('/workspaces/Project-Option-1-Matt-Flores')

---               
# Dataset Review

Now that the data is loaded we can review and profile each table loaded into the directory.  
There are 5 raw data files loaded into the data folder.
1) **movies** : This is the main df because the primary key, ***movieId***, in this df is the projects **unit of analysis**.  

2) **ratings** : This dataset contains almost 100k reviews throughout 1995 to 2016. This data will need to be aggregated to be merged with the movies table. 

3) **links** : Contains the imdbId & tmdbId associated with each movieid. It allows the ratings df to be joined with the credits and keywords tables without having to go through the movie dataset, which contains all 3 id types.  

4) **credits** : Lists the directors and top billed actors in each film. Maybe there is a correlation between revenue and directors/actors.  

5) **keywords** : Contains The Movie Database ID along with a multi-valued field listing descriptive words for each movie. Genre is a broad categorization of a film. keywords df can provide more detail on themes, locations, movie plots, characters, etc.


In [3]:
#This code loads the raw data into dataframes. And creates a column summary by table df.
# Load the datasets
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

# Create a dictionary to hold the DataFrames for easier access. We can loop through tables, etc.
tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

# Collect the column names for each table
# for each table in tables dictionary, append the table name to the first column 
# and then each column name in the table to the next columns.
summary_data = []
for name, df in tables.items():
    summary_data.append({"table_name": name, **{f"col_{i+1}": col for i, col in enumerate(df.columns)}})

# Build a summary DataFrame that shows the table name and each column name in separate columns.
column_summary_df = pd.DataFrame(summary_data).set_index("table_name")

# Display the result
display(column_summary_df)

,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_20,col_21
table_name,,,,,,,,,,,,,,,,,,,,,
movies,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
ratings,userId,movieId,rating,timestamp,rating_datetime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
links,movieId,imdbId,tmdbId,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credits,tmdbId,director,top_cast,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keywords,tmdbId,keywords_list,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#Initial data profiling: What are the columns, data types, and missing values for each table?

for name, df in tables.items():
    print(f"\n--- {name} Profile Summary ---")
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    
    # 1. Create a summary DataFrame containing dtypes and missing rates
    summary_df = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_rate': df.isna().mean(),
        'missing_count': df.isna().sum()
    })
    
    # 2. Display the combined table
    display(summary_df.sort_values(by='missing_rate', ascending=False))


--- movies Profile Summary ---

movies: 9,082 rows x 21 columns


,dtype,missing_rate,missing_count
genres_list,object,0.003854,35
primary_genre,object,0.003854,35
status,object,0.000220,2
tmdbId,int64,0.000000,0
movieId,int64,0.000000,0
original_title,object,0.000000,0
title,object,0.000000,0
imdb_id,object,0.000000,0
original_language,object,0.000000,0
revenue,float64,0.000000,0



--- ratings Profile Summary ---

ratings: 99,810 rows x 5 columns


,dtype,missing_rate,missing_count
userId,int64,0.0,0
movieId,int64,0.0,0
rating,float64,0.0,0
timestamp,int64,0.0,0
rating_datetime,datetime64[ns],0.0,0



--- links Profile Summary ---

links: 9,125 rows x 3 columns


,dtype,missing_rate,missing_count
tmdbId,float64,0.001425,13
movieId,int64,0.000000,0
imdbId,int64,0.000000,0



--- credits Profile Summary ---

credits: 9,082 rows x 3 columns


,dtype,missing_rate,missing_count
top_cast,object,0.009910,90
director,object,0.002643,24
tmdbId,int64,0.000000,0



--- keywords Profile Summary ---

keywords: 9,082 rows x 2 columns


,dtype,missing_rate,missing_count
keywords_list,object,0.083792,761
tmdbId,int64,0.000000,0


In [5]:
## In this code I examine the movies revenue. This project focuses on the primary unit of data, which is movies. This table is most important.
# limit column width so the text doesnt wrap and make the rows wide.
pd.set_option('display.max_colwidth',12)

# Drop only the columns that cause wide text wrapping
movie_cols_to_show = [col for col in movies.columns if col not in ["genres", "production_companies", "production_countries"]]
#display(movie_cols_to_show)

# Display top 3 movies by revenue
display(
    movies.sort_values(by="revenue", ascending=False).head(3)
)

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
7390,72998,19995,tt0499549,Avatar,Avatar,en,2009-12-10,162.0,237000000,2.787965...,185.070892,7.2,12114.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2009
8746,122886,140607,tt2488496,Star War...,Star War...,en,2015-12-15,136.0,245000000,2.068224...,31.626013,7.5,7993.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2015
1355,1721,597,tt0120338,Titanic,Titanic,en,1997-11-18,194.0,200000000,1.845034...,26.889070,7.5,7770.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Drama|Ro...,Drama,1997


In [6]:
# Display top 3 movies by release date desc. Release date is string, but sorting still works on it bc it's in numerical order.
display(movies.sort_values(by="release_date", ascending=False)[["movieId", "title", "release_date"]].head(3))

# Convert release_date to actual datetime objects, then sort
movies_sorted = movies.copy()
movies_sorted['release_date'] = pd.to_datetime(movies_sorted['release_date'], errors='coerce')

display(
    movies_sorted.sort_values(by="release_date", ascending=False)[["movieId", "title", "release_date"]].head(3)
)

#Sorting by date has same result. Most recent movie is september 2023.

,movieId,title,release_date
9023,151307,The Love...,2016-09-23
9081,163949,The Beat...,2016-09-15
8747,122888,Ben-Hur,2016-08-17


,movieId,title,release_date
9023,151307,The Love...,2016-09-23
9081,163949,The Beat...,2016-09-15
8747,122888,Ben-Hur,2016-08-17


In [22]:
display(ratings['movieId'].drop_duplicates())

display(ratings['rating_datetime'].sort_values(ascending=False).iloc[0])
display(ratings['rating_datetime'].sort_values(ascending=True).iloc[0])

display(movies['release_date'].sort_values(ascending=False).iloc[0])
display(movies['release_date'].sort_values(ascending=True).iloc[0])

0           31
1         1029
2         1061
3         1129
4         1172
         ...  
98938    64997
98966    72380
99081      129
99485     4736
99626     6425
Name: movieId, Length: 9025, dtype: int64

Timestamp('2016-10-16 17:57:24')

Timestamp('1995-01-09 11:46:49')

'2016-09-23'

'1902-09-01'

In [9]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,object
title,object
original_title,object
original_language,object
release_date,object
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[ns]


...

,dtype
tmdbId,int64
director,object
top_cast,object


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,object


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


In [85]:
for name, df in tables.items():
    print(f"\n--- {name} Profile Summary ---")
    
    # 1. Create a summary DataFrame containing dtypes and missing rates
    summary_df = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_rate': df.isna().mean(),
        'missing_count': df.isna().sum()
    })
    
    # 2. Display the combined table
    display(summary_df.sort_values(by='missing_rate', ascending=False))


--- movies Profile Summary ---


,dtype,missing_rate,missing_count
genres_list,str,0.003854,35
primary_genre,str,0.003854,35
status,str,0.000220,2
tmdbId,int64,0.000000,0
movieId,int64,0.000000,0
original_title,str,0.000000,0
title,str,0.000000,0
imdb_id,str,0.000000,0
original_language,str,0.000000,0
revenue,float64,0.000000,0



--- ratings Profile Summary ---


,dtype,missing_rate,missing_count
userId,int64,0.0,0
movieId,int64,0.0,0
rating,float64,0.0,0
timestamp,int64,0.0,0
rating_datetime,datetime...,0.0,0



--- links Profile Summary ---


,dtype,missing_rate,missing_count
tmdbId,float64,0.001425,13
movieId,int64,0.000000,0
imdbId,int64,0.000000,0



--- credits Profile Summary ---


,dtype,missing_rate,missing_count
top_cast,str,0.009910,90
director,str,0.002643,24
tmdbId,int64,0.000000,0



--- keywords Profile Summary ---


,dtype,missing_rate,missing_count
keywords_list,str,0.083792,761
tmdbId,int64,0.000000,0


In [80]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Animatio...,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Adventur...,Adventure,1995
2,3,15602,tt0113228,Grumpier...,Grumpier...,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Romance|...,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-1...
1,1,1029,3.0,1260759179,2009-12-1...
2,1,1061,3.0,1260759182,2009-12-1...



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Las...,Tom Hank...
1,8844,Joe John...,Robin Wi...
2,15602,Howard D...,Walter M...



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy...
1,8844,board ga...
2,15602,fishing|...


In [8]:
print("--- Movies ---")
display(movies[movies["movieId"] == 1])

print("--- Ratings ---")
display(ratings[ratings["movieId"] == 1])

print("--- Links ---")
display(links[links["movieId"] == 1])

print("--- Credits ---")
display(credits[credits["tmdbId"] == 862])

print("--- Keywords ---")
display(keywords[keywords["tmdbId"] == 862])

display(movies.sort_values(by="revenue", ascending=False).head(10))

display(movies.sort_values(by="release_date", ascending=False).head(10))

movies[movies.isna().any(axis=1)]

display(movies[movies.isna().any(axis=1) & (movies["revenue"] != 0)])

# Group by movieId to get the average, median, and total rating count
rating_summary = ratings.groupby("movieId").agg(
    rating_avg=("rating", "mean"),
    rating_median=("rating", "median"),
    rating_count=("userId", "count")
).reset_index()

print("--- Merge Rating Summary to Movie df ---")
rating_summary_with_titles = rating_summary.merge(
    movies[["movieId", "title", "vote_average", "vote_count"]], 
    on="movieId", 
    how="left"
)

# Display top 100 with titles
display(rating_summary_with_titles.sort_values(by="rating_count", ascending=False).head(100))

--- Movies ---


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995


--- Ratings ---


,userId,movieId,rating,timestamp,rating_datetime
494,7,1,3.0,851866703,1996-12-29 13:38:23
697,9,1,4.0,938629179,1999-09-29 18:19:39
887,13,1,5.0,1331380058,2012-03-10 11:47:38
959,15,1,2.0,997938310,2001-08-16 05:05:10
3099,19,1,3.0,855190091,1997-02-06 00:48:11
...,...,...,...,...,...
98341,660,1,2.5,1436680062,2015-07-12 05:47:42
98523,663,1,4.0,1438397999,2015-08-01 02:59:59
98549,664,1,3.5,1362421730,2013-03-04 18:28:50
99664,670,1,4.0,938782344,1999-10-01 12:52:24


--- Links ---


...

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
7390,72998,19995,tt0499549,Avatar,Avatar,en,2009-12-10,162.0,237000000,2.787965e+09,185.070892,7.2,12114.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Ingenious Film Partners', 'id': 289...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Fantasy|Science Fiction,Action,2009
8746,122886,140607,tt2488496,Star Wars: The Force Awakens,Star Wars: The Force Awakens,en,2015-12-15,136.0,245000000,2.068224e+09,31.626013,7.5,7993.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Lucasfilm', 'id': 1}, {'name': 'Tru...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Science Fiction|Fantasy,Action,2015
1355,1721,597,tt0120338,Titanic,Titanic,en,1997-11-18,194.0,200000000,1.845034e+09,26.889070,7.5,7770.0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama|Romance|Thriller,Drama,1997
7864,89745,24428,tt0848228,The Avengers,The Avengers,en,2012-04-25,143.0,220000000,1.519558e+09,89.887648,7.4,12000.0,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Science Fiction|Action|Adventure,Science Fiction,2012
8700,117529,135397,tt0369610,Jurassic World,Jurassic World,en,2015-06-09,124.0,150000000,1.513529e+09,32.790475,6.5,8842.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Universal Studios', 'id': 13}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Science Fiction|Thriller,Action,2015
8820,130634,168259,tt2820852,Furious 7,Furious 7,en,2015-04-01,137.0,190000000,1.506249e+09,27.275687,7.3,4253.0,"[{'id': 28, 'name': 'Action'}]","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'JP', 'name': 'Japan'}, {'iso_...",False,Released,Action,Action,2015
8749,122892,99861,tt2395427,Avengers: Age of Ultron,Avengers: Age of Ultron,en,2015-04-22,141.0,280000000,1.405404e+09,37.379420,7.3,6908.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Marvel Studios', 'id': 420}, {'name...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Science Fiction,Action,2015
7816,88125,12445,tt1201607,Harry Potter and the Deathly Hallows: Part 2,Harry Potter and the Deathly Hallows: Part 2,en,2011-07-07,130.0,125000000,1.342000e+09,24.990737,7.9,6141.0,"[{'id': 10751, 'name': 'Family'}, {'id': 14, '...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Family|Fantasy|Adventure,Family,2011
8429,106696,109445,tt2294629,Frozen,Frozen,en,2013-11-27,102.0,150000000,1.274219e+09,24.248243,7.3,5440.0,"[{'id': 16, 'name': 'Animation'}, {'id': 12, '...","[{'name': 'Walt Disney Pictures', 'id': 2}, {'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Adventure|Family,Animation,2013
8280,102125,68721,tt1300854,Iron Man 3,Iron Man 3,en,2013-04-18,130.0,200000000,1.215440e+09,23.721243,6.8,8951.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Marvel Studios', 'id': 420}]","[{'iso_3166_1': 'CN', 'name': 'China'}, {'iso_...",False,Released,Action|Adventure|Science Fiction,Action,2013


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
9023,151307,373355,tt5278868,The Lovers and the Despot,The Lovers and the Despot,en,2016-09-23,100.0,0,0.0,0.744523,7.0,4.0,"[{'id': 99, 'name': 'Documentary'}]",[],[],False,Released,Documentary,Documentary,2016
9081,163949,391698,tt2531318,The Beatles: Eight Days a Week - The Touring Y...,The Beatles: Eight Days a Week - The Touring Y...,en,2016-09-15,99.0,0,0.0,7.078301,7.6,92.0,"[{'id': 99, 'name': 'Documentary'}, {'id': 104...","[{'name': 'Imagine Entertainment', 'id': 23}, ...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Documentary|Music,Documentary,2016
8747,122888,271969,tt2638144,Ben-Hur,Ben-Hur,en,2016-08-17,125.0,100000000,94061311.0,11.510210,5.3,642.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Drama|Action,Adventure,2016
9073,161582,338766,tt2582782,Hell or High Water,Hell or High Water,en,2016-08-12,102.0,12000000,37589296.0,12.565896,7.2,1304.0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'name': 'Sidney Kimmel Entertainment', 'id':...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Crime|Drama|Thriller|Western,Crime,2016
9078,162542,392572,tt5165344,Rustom,रुस्तम,hi,2016-08-12,150.0,1000000,0.0,7.333139,7.3,25.0,"[{'id': 53, 'name': 'Thriller'}, {'id': 10749,...","[{'name': 'KriArj Entertainment', 'id': 91689}]","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Thriller|Romance,Thriller,2016
9079,162672,402672,tt3859980,Mohenjo Daro,Mohenjo Daro,hi,2016-08-11,155.0,15050000,16180000.0,1.423358,6.7,26.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'UTV Motion Pictures', 'id': 2320}, ...","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Adventure|Drama|History|Romance,Adventure,2016
8885,135536,297761,tt1386697,Suicide Squad,Suicide Squad,en,2016-08-02,123.0,175000000,745600054.0,42.965027,5.9,7717.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'DC Comics', 'id': 429}, {'name': 'D...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Crime|Fantasy|Science Fiction,Action,2016
9076,161918,390989,tt4831420,Sharknado 4: The 4th Awakens,Sharknado 4: The 4th Awakens,en,2016-07-31,85.0,0,0.0,4.574494,4.3,88.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'The Asylum', 'id': 1311}, {'name': ...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Horror|Science Fiction,Comedy,2016
9080,163056,315011,tt4262980,Shin Godzilla,シン・ゴジラ,ja,2016-07-29,120.0,15000000,77000000.0,9.285519,6.6,152.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Cine Bazar', 'id': 5896}, {'name': ...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",False,Released,Action|Adventure|Drama|Horror|Science Fiction,Action,2016
9061,160438,324668,tt4196776,Jason Bourne,Jason Bourne,en,2016-07-27,123.0,120000000,415484914.0,19.133256,5.9,2386.0,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...","[{'name': 'The Kennedy/Marshall Company', 'id'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Action|Thriller,Action,2016


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
4544,6269,51927,tt0334416,Stevie,Stevie,en,2002-09-09,140.0,0,97000.0,0.489997,6.7,13.0,"[{'id': 99, 'name': 'Documentary'}, {'id': 18,...",[],[],False,NaN,Documentary|Drama|Foreign,Documentary,2002
5470,8622,1777,tt0361596,Fahrenheit 9/11,Fahrenheit 9/11,en,2004-06-25,122.0,6000000,119114517.0,6.839460,6.9,403.0,[],"[{'name': 'BIM Distribuzione', 'id': 225}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,NaN,NaN,2004


--- Merge Rating Summary to Movie df ---


,movieId,rating_avg,rating_median,rating_count,title,vote_average,vote_count
321,356,4.054252,4.0,341,Forrest Gump,8.2,8147.0
266,296,4.256173,4.5,324,Pulp Fiction,8.3,8670.0
284,318,4.487138,5.0,311,The Shawshank Redemption,8.5,8358.0
525,593,4.138158,4.0,304,The Silence of the Lambs,8.1,4549.0
232,260,4.221649,4.5,291,Star Wars,8.1,6778.0
...,...,...,...,...,...,...,...
6872,58559,4.235537,4.5,121,The Dark Knight,8.3,12269.0
1902,2396,3.966942,4.0,121,Shakespeare in Love,6.8,831.0
958,1206,4.000000,4.0,121,A Clockwork Orange,8.0,3432.0
37,39,3.550000,3.5,120,Clueless,6.9,828.0
